In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from sci_helpers import csv_to_samples_list
from sci_helpers import compare_wrangled

In [2]:
# Example call (adjust filepath):
s_list, ox_order, df_num = csv_to_samples_list('Aplites_HAL_XRF_noUncertainty.csv', filepath=r'C:\Users\marvi\PycharmProjects\sci-cluster\sci-cluster\sci-data\Aplites_XRF_data')
print(ox_order)
print(s_list[0])
print(df_num)

['SiO2', 'TiO2', 'Al2O3', 'FeO*', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
['KCP109C', np.float64(75.82), np.float64(0.082), np.float64(12.95), np.float64(0.66), np.float64(0.031), np.float64(0.09), np.float64(0.7), np.float64(4.37), np.float64(4.01), np.float64(0.009)]
             KCP109C  KCP109B  KCP109A  KHD-106-A  KHD-105-G  KHD-105-D  \
Sample Name                                                               
SiO2          75.820   76.360   76.620     75.800     76.230     76.070   
TiO2           0.082    0.069    0.079      0.131      0.058      0.092   
Al2O3         12.950   12.650   12.480     12.900     13.050     13.070   
FeO*           0.660    0.520    0.620      1.100      0.450      0.640   
MnO            0.031    0.024    0.026      0.035      0.005      0.008   
MgO            0.090    0.050    0.060      0.160      0.050      0.080   
CaO            0.700    0.540    0.530      1.060      0.960      1.040   
Na2O           4.370    3.970    3.980      3.210  

Below is a test call of the function 'csv_to_samples_list' , but applied to my aplite dataset. 

In [3]:
s_list, ox_order, df_num = csv_to_samples_list('Aplites_HAL_XRF_noUncertainty.csv',filepath=r'C:\Users\marvi\PycharmProjects\sci-cluster\sci-cluster\sci-data\Aplites_XRF_data')
print(ox_order)
print(s_list[0])
print(df_num)

['SiO2', 'TiO2', 'Al2O3', 'FeO*', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
['KCP109C', np.float64(75.82), np.float64(0.082), np.float64(12.95), np.float64(0.66), np.float64(0.031), np.float64(0.09), np.float64(0.7), np.float64(4.37), np.float64(4.01), np.float64(0.009)]
             KCP109C  KCP109B  KCP109A  KHD-106-A  KHD-105-G  KHD-105-D  \
Sample Name                                                               
SiO2          75.820   76.360   76.620     75.800     76.230     76.070   
TiO2           0.082    0.069    0.079      0.131      0.058      0.092   
Al2O3         12.950   12.650   12.480     12.900     13.050     13.070   
FeO*           0.660    0.520    0.620      1.100      0.450      0.640   
MnO            0.031    0.024    0.026      0.035      0.005      0.008   
MgO            0.090    0.050    0.060      0.160      0.050      0.080   
CaO            0.700    0.540    0.530      1.060      0.960      1.040   
Na2O           4.370    3.970    3.980      3.210  

## More tests below

In [4]:
import importlib
import sci_helpers
importlib.reload(sci_helpers)
# also reload the submodule directly if needed
import sci_helpers.stacked_to_samples as sts
importlib.reload(sts)
from sci_helpers.stacked_to_samples import stacked_file_to_wrangled, merge_wrangled_results

### Test of loading a sheet on Monte Carlo synthetic compositions

In [5]:
# import (either direct submodule import or package-level import now that we exported it)
from sci_helpers.stacked_to_samples import stacked_file_to_wrangled, merge_wrangled_results

path = r"C:\Users\marvi\PycharmProjects\sci-cluster\sci-cluster\sci-data\Aplites_XRF_data\MonteCarlo_synthetic_compositions.csv"

# Proper multi-line call (no inline comments inside the parentheses)
results = stacked_file_to_wrangled(
    path,
    anchor='SiO2',
    skip_first_cols=6,        # skip left A..F metadata -> start at G
    synthetic_rows=200,       # take 200 rows after each synthetic header
    auto_detect_skip=False,
    blank_row_tolerance=50
)

# iterate all detected stacked datasets and show them
import os
from IPython.display import display

print("datasets found:", len(results))

dataset_frames = {}  # name -> df (with sample names as index)
for i, ds in enumerate(results, start=1):
    # name from Tab Names column (or fallback)
    raw_name = ds.get('dataset_name')
    name = raw_name if raw_name and raw_name.strip() else f'dataset_{i}'
    samples = ds.get('samples_list', [])
    oxides = ds.get('oxides_order', [])
    notes = ds.get('notes', [])

    print(f"--- Dataset {i} : {name} ---")
    print(" anchor (row,col):", ds.get('anchor'))
    print(" col_offset:", ds.get('col_offset'))
    print(" #samples:", len(samples))
    print(" oxides_order:", oxides)
    print(" notes (first 5):", notes[:5])
    if samples:
        print(" first sample row (0):", samples[0])
    else:
        print(" (no samples found)")

    # Build a tidy DataFrame view for this dataset and display it
    df_block = ds.get('df')
    try:
        # set index from samples_list sample names (preserve order)
        sample_index = [str(s[0]) for s in samples]
        df_view = df_block.copy()
        df_view.index = sample_index
    except Exception:
        df_view = df_block.copy()
    display(df_view.head())

    # store for later merging/export
    dataset_frames[name] = df_view


datasets found: 17
--- Dataset 1 : KCP-109-C ---
 anchor (row,col): (1, 1)
 col_offset: 6
 #samples: 200
 oxides_order: ['SiO2', 'TiO2', 'Al2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
 notes (first 5): ['Detected synthetic header at row 1, columns 1-10', 'WROTE_CSV: C:\\Users\\marvi\\PycharmProjects\\sci-cluster\\sci-cluster\\sci-data\\wrangled-outputs\\wrangled_KCP-109-C_compositions.csv']
 first sample row (0): ['KCP-109-B', 76.28822443, 0.059669847, 13.04695036, 0.643322296, 0.035780297, 0.101281661, 0.642557927, 4.489726469, 3.954553845, 0.021808209]


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
KCP-109-B,76.288224,0.059670,13.046950,0.643322,0.035780,0.101282,0.642558,4.489726,3.954554,0.021808
KCP-109-A,76.129650,0.075127,12.908501,0.667325,0.026542,0.127045,0.710470,4.316284,4.063871,-0.002402
KCP-110-A,74.962688,0.061927,12.960715,0.656584,0.027906,0.080919,0.659863,4.401716,3.982320,0.008698
KHD-106-A,75.501964,0.065099,12.845957,0.655491,0.032188,0.097903,0.708541,4.193391,3.938005,0.013951
KHD-105-G,75.391956,0.087528,13.068616,0.697873,0.033906,0.135814,0.742119,4.360707,4.133836,0.010315


--- Dataset 2 : KCP-109-B ---
 anchor (row,col): (208, 1)
 col_offset: 6
 #samples: 200
 oxides_order: ['SiO2', 'TiO2', 'Al2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
 notes (first 5): ['Detected synthetic header at row 208, columns 1-10', 'WROTE_CSV: C:\\Users\\marvi\\PycharmProjects\\sci-cluster\\sci-cluster\\sci-data\\wrangled-outputs\\wrangled_KCP-109-B_compositions.csv']
 first sample row (0): ['nan', 76.5013638, 0.065759029, 12.45939442, 0.518972786, 0.024787721, 0.072155292, 0.570261896, 3.903717416, 4.530558631, 0.005839597]


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
nan,76.501364,0.065759,12.459394,0.518973,0.024788,0.072155,0.570262,3.903717,4.530559,0.005840
nan,76.753695,0.077744,12.425437,0.528506,0.026818,0.033276,0.563422,3.672927,4.584184,0.007011
nan,76.335021,0.059368,12.553869,0.479527,0.021943,0.026918,0.522082,4.158732,4.671548,0.007514
nan,76.686630,0.057436,12.726127,0.550799,0.022482,0.025945,0.520658,4.151702,4.540391,-0.002878
nan,77.262050,0.065859,12.623770,0.490982,0.023843,0.056295,0.529913,4.059493,4.530927,0.001171


--- Dataset 3 : KCP-109-A ---
 anchor (row,col): (415, 1)
 col_offset: 6
 #samples: 200
 oxides_order: ['SiO2', 'TiO2', 'Al2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
 notes (first 5): ['Detected synthetic header at row 415, columns 1-10', 'WROTE_CSV: C:\\Users\\marvi\\PycharmProjects\\sci-cluster\\sci-cluster\\sci-data\\wrangled-outputs\\wrangled_KCP-109-A_compositions.csv']
 first sample row (0): ['nan', 76.67049527, 0.07269311, 12.56150844, 0.61610213, 0.028580365, 0.0538821, 0.489936128, 3.920815604, 4.50988437, -0.000890598]


...

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
nan,75.883978,-0.031760,12.927542,0.941444,0.027485,-0.070997,1.032798,2.966608,5.882109,0.037594
nan,75.550830,0.081111,13.134925,0.532107,0.016126,0.735117,0.929872,3.012012,5.978073,-0.051594
nan,75.294125,0.065658,13.344581,0.742842,0.011035,0.094410,0.535229,3.009136,5.874636,0.068524
nan,75.520040,0.161251,13.228167,0.796564,0.016486,-0.342415,0.914685,2.770607,5.925962,-0.046687
nan,74.859716,0.108441,13.297416,1.058667,0.030481,0.618568,0.756934,3.160182,5.930469,0.069242


--- Dataset 16 : KHD-105-B-2 ---
 anchor (row,col): (3111, 1)
 col_offset: 6
 #samples: 200
 oxides_order: ['SiO2', 'TiO2', 'Al2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
 notes (first 5): ['Detected synthetic header at row 3111, columns 1-10', 'WROTE_CSV: C:\\Users\\marvi\\PycharmProjects\\sci-cluster\\sci-cluster\\sci-data\\wrangled-outputs\\wrangled_KHD-105-B-2_compositions.csv']
 first sample row (0): ['nan', 75.81826017, 0.094469475, 13.01280919, 0.754342726, 0.010480288, 0.107755815, 0.919397818, 2.81130534, 5.644436196, 0.014738099]


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
nan,75.818260,0.094469,13.012809,0.754343,0.010480,0.107756,0.919398,2.811305,5.644436,0.014738
nan,76.677459,0.090026,13.025951,0.830653,0.014227,0.104769,0.974588,2.941506,5.686728,0.011531
nan,76.400974,0.084808,12.691868,0.850459,0.004085,0.142696,0.993341,3.023797,5.721332,0.011278
nan,76.127229,0.119005,13.019009,0.900363,0.008099,0.130000,0.949643,2.934279,5.671418,0.016348
nan,76.746774,0.088971,12.881043,0.915008,0.014195,0.089491,0.910613,2.761615,5.697037,0.015633


--- Dataset 17 : TEMPLATE1 ---
 anchor (row,col): (3318, 1)
 col_offset: 6
 #samples: 200
 oxides_order: ['SiO2', 'TiO2', 'Al2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O', 'P2O5']
 notes (first 5): ['Detected synthetic header at row 3318, columns 1-10', 'WROTE_CSV: C:\\Users\\marvi\\PycharmProjects\\sci-cluster\\sci-cluster\\sci-data\\wrangled-outputs\\wrangled_TEMPLATE1_compositions.csv']
 first sample row (0): ['nan', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5
nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import inspect
print(inspect.signature(stacked_file_to_wrangled))

(path: str, sheet_name: Optional[str] = None, anchor: str = 'SiO2', oxide_names: Optional[List[str]] = None, min_oxides: int = 3, blank_row_tolerance: int = 50, auto_name_prefix: str = 'sample_auto_', write_out: Optional[str] = None, skip_first_cols: int = 0, auto_detect_skip: bool = False, synthetic_rows: Optional[int] = None, header_scan_rows: Optional[int] = None, tab_name_col: int = 0) -> List[Dict]
